In [1]:
import csv
import argparse
import os
from google.colab import files
import re
import zipfile
from openai import OpenAI
import json
import threading
from threading import Semaphore

# Hypothesis and Ground Truth data loading and check

In [2]:
class TSVLoader:
    def __init__(self, source_path, target_path, hyp_source_path, hyp_target_path, has_header=False):
        self.source_path = source_path
        self.target_path = target_path
        self.hyp_source_path = hyp_source_path
        self.hyp_target_path = hyp_target_path
        self.has_header = has_header

        # Lists that will store loaded lines
        self.source = []
        self.target = []
        self.hyp_source = []
        self.hyp_target = []

    def load(self):
        # Load source.tsv
        with open(self.source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.source = lines

        # Load target.tsv
        with open(self.target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.target = lines

        # Load hyp_source.tsv
        with open(self.hyp_source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_source = lines

        # Load hyp_target.tsv
        with open(self.hyp_target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_target = lines

    def print_statistics(self):
        print(f"Loaded {len(self.source)} source lines.")
        print(f"Loaded {len(self.target)} target lines.")
        print(f"Loaded {len(self.hyp_source)} hyp_source lines.")
        print(f"Loaded {len(self.hyp_target)} hyp_target lines.")

In [3]:
uploaded = files.upload()

Saving hyp_target.tsv to hyp_target.tsv
Saving hyp_source.tsv to hyp_source.tsv
Saving target.tsv to target.tsv
Saving source.tsv to source.tsv


In [4]:
loader = TSVLoader(
    "source.tsv",
    "target.tsv",
    "hyp_source.tsv",
    "hyp_target.tsv",
    has_header=False
)

loader.load()
loader.print_statistics()

source = loader.hyp_source
machineTranslation = loader.hyp_target

Loaded 2 source lines.
Loaded 2 target lines.
Loaded 2 hyp_source lines.
Loaded 2 hyp_target lines.


In [5]:
source

['Tom is still willing to do that for free.', 'Tom bought Mary something.']

In [6]:
machineTranslation

['Tom ainda está disposto a fazer isso de graça.',
 'Tom comprou algo para a Mary.']

# Prompts

In [7]:
class Prompts:

    def upload_and_unzip(self):
        """Upload a zip file and unzip it in the current working directory."""
        print("Please upload your ZIP file:")
        uploaded = files.upload()
        zip_filename = list(uploaded.keys())[0]

        # Extract to a folder named after the zip (without extension)
        extract_dir = os.path.splitext(zip_filename)[0]
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(f"Files extracted to: {extract_dir}")
        return extract_dir


    def load_text_files(self, directory):
        """
        Load all .txt files in directory and return a nested dictionary
        data[type1][type2] = file contents
        """
        data = {}

        for root, _, files_in_dir in os.walk(directory):
            for fname in files_in_dir:
                if fname.endswith(".txt"):
                    parts = fname.replace(".txt", "").split("_")
                    # Expect pattern: system_<type1>_<type2>.txt
                    if len(parts) >= 3:
                        type1 = parts[1]
                        type2 = parts[2]
                        fpath = os.path.join(root, fname)

                        with open(fpath, "r", encoding="utf-8") as f:
                            content = f.read()

                        if type1 not in data:
                            data[type1] = {}

                        data[type1][type2] = content

        return data


    def fill_template(self, text, variables):
        """Replace {var} placeholders using variables dict."""
        def replacer(match):
            key = match.group(1)  # get variable name inside {}
            return str(variables.get(key, f"{{{key}}}"))  # keep {key} if missing

        # Find {variable_name} patterns
        return re.sub(r"\{(\w+)\}", replacer, text)

In [8]:
prompts_obj = Prompts()

# Upload e unzip
folder = prompts_obj.upload_and_unzip()

# Ler arquivos de texto
data = prompts_obj.load_text_files(folder)

Please upload your ZIP file:


Saving Prompts.zip to Prompts.zip
Files extracted to: Prompts


In [9]:
data

{'term': {'inappropriate': 'You are a Translation Terminology Evaluation Agent. Detect only terminology errors: error occurring when the translator does not use the required or standardized term from a specified termbase or other reference resource, even if the overall meaning of the sentence remains correct.\nEsse tipo de erro é diferente de um erro de tradução (mistranslation), que altera o significado da frase. O erro de terminologia foca na conformidade com a terminologia aprovada, e não na precisão semântica.\nExemplos:\nA termbase especifica que “Corporate Income Tax” deve sempre ser traduzido como “Imposto sobre a Renda da Pessoa Jurídica (IRPJ)”, mas a tradução diz “Imposto de Renda Empresarial”.\n(O significado é parecido, mas não segue o termo oficial definido.)\nO texto-fonte diz “Click Submit to send the form.”, e a termbase define “Submit → Enviar formulário”, mas a tradução diz “Clique em Enviar”.\n(O sentido está correto, mas o termo padronizado não foi seguido.)\n',
  '

# Metadata

In [10]:
metas = {}
metas["acc"] = {}
metas["acc"]["pt_sup_error"] = "acurácia"
metas["acc"]["addition"] = "adição"
metas["acc"]["mistranslation"] = "tradução incorreta"
metas["acc"]["name"] = "consistência de nomes"
metas["acc"]["untranslated"] = "não tradução"
metas["acc"]["omission"] = "omissão"


metas["linc"] = {}
metas["linc"]["pt_sup_error"] = "convenção linguística"
metas["linc"]["grammar"] = "gramática"
metas["linc"]["punctuation"] = "pontuação"
metas["linc"]["spelling"] = "ortografia incorreta"
metas["linc"]["character"] = "codificação de caracteres"

metas["loc"] = {}
metas["loc"]["pt_sup_error"] = "convenção do país"
metas["loc"]["time"] = "formato de hora"
metas["loc"]["monetary"] = "convenção monetária"

metas["sty"] = {}
metas["sty"]["pt_sup_error"] = "estilo incompatível"
metas["sty"]["awkward"] = "texto não natural"

metas["term"] = {}
metas["term"]["pt_sup_error"] = "terminologia"
metas["term"]["inappropriate"] = "terminologia inapropriada"
metas["term"]["inconsistent"] = "inconsistência de terminologia"

# Client

In [ ]:
KEY = 'your_openai_api_key_here'
MODEL_OPENAI = 'gpt-5-nano'
REASONING = 'minimal'
VERBOSITY = 'low'
os.environ["OPENAI_API_KEY"] = KEY
client = OpenAI()

In [12]:
class GPTClient:
    def __init__(self, model="gpt-5", verbosity="low", reasoning="low"):
        self.model = model
        self.verbosity = verbosity
        self.reasoning = reasoning
        self.history = []

    def appendHistory(self, history,role,prompt):
      field = "input_text"
      if (role=="assistant"):
        field = "output_text"
      content = {
            "role": role,
            "content": [
                {
                "type": field,
                "text": prompt
                }
            ]
      }
      history.append(content)
      return history


    def callGPT(self, history):
        response = client.responses.create(
            model=MODEL_OPENAI,
            input=history,
            text={
                "format": {
                "type": "json_object"
                },
                "verbosity": VERBOSITY
            },
            reasoning={
                "effort": REASONING
            },
            tools=[],
            store=False,
            include=[
            ]
        )
        raw_output = response.output[1].content[0].text

        parsed_output = json.loads(raw_output)
        return parsed_output

# Agents

In [13]:
class Specialist(GPTClient):
    def __init__(self, prompts, data, metas, model="gpt-5", verbosity="low", reasoning="default"):
        super().__init__(model=model, verbosity=verbosity, reasoning=reasoning)
        self.prompts = prompts
        self.data = data
        self.metas = metas

    def SpecialistEvaluation(self, source,sp_vars, machineTranslation, type1, type2, printInfo=True):


        # Prepare initial Specialist prompt
        specialistEvaluation = self.prompts.fill_template(
            self.data["evaluation"]["first"], sp_vars
        )

        #if printInfo:
            #print(self.data[type1][type2])
            #print("SP:", specialistEvaluation)

        # Build conversation history
        sphistory = []
        sphistory = self.appendHistory(sphistory, "developer", self.data[type1][type2])
        sphistory = self.appendHistory(sphistory, "user", specialistEvaluation)

        specialistResponse = self.callGPT(sphistory)

        #if printInfo:
            #print("Severidade:", specialistResponse.get("erro"))
            #print("Raciocinio:", specialistResponse.get("raciocinio"))
            #print("Proposta:", specialistResponse.get("proposta"))

        return specialistResponse


    def SelfReflection(self, source, machineTranslation, type1, type2, sp_vars, specialistResponse, printInfo):

        # Fill specialist-derived info
        sp_vars["severidade"] = specialistResponse.get("erro")
        sp_vars["raciocinio"] = specialistResponse.get("raciocinio") or specialistResponse.get("raciocínio")
        sp_vars["proposta"] = specialistResponse.get("proposta")

        # Build self-reflection prompt
        selfreflection_prompt = self.prompts.fill_template(
            self.data["evaluation"]["selfreflection"], sp_vars
        )

        # Prepare history for self-reflection
        reflectionHistory = []
        reflectionHistory = self.appendHistory(reflectionHistory, "developer", self.data[type1][type2])
        reflectionHistory = self.appendHistory(reflectionHistory, "user", selfreflection_prompt)

        # Call GPT
        selfReflectionResponse = self.callGPT(reflectionHistory)

        return selfReflectionResponse

In [14]:
class Supervisor(GPTClient):
    def __init__(self, prompts, data, model="gpt-5", verbosity="low", reasoning="low"):
        super().__init__(model=model, verbosity=verbosity, reasoning=reasoning)
        self.prompts = prompts
        self.data = data


    def SupervisorEvaluation(self, type1, sp_vars, printInfo=True):
        """
        Runs the supervisor-level evaluation based on the variables
        generated by the Specialist stage (sp_vars).
        """

        # Build the supervisor prompt
        supervisor_prompt = self.prompts.fill_template(
            self.data["evaluation"]["supervisor"],
            sp_vars
        )

        # Build conversation history
        supervisorHistory = []
        supervisorHistory = self.appendHistory(
            supervisorHistory,
            "developer",
            self.data[type1]["supervisor"]
        )
        supervisorHistory = self.appendHistory(
            supervisorHistory,
            "user",
            supervisor_prompt
        )

        #if printInfo:
            #print("SupervisorHistory:", supervisorHistory)

        # Call the model
        supervisorResponse = self.callGPT(supervisorHistory)

        #if printInfo:
            #print("SupervisorResponse:", supervisorResponse)

        return supervisorResponse

In [15]:
class Orchestrator(GPTClient):
    def __init__(self, prompts, data, metas,specialist_agent, model="gpt-5", verbosity="low", reasoning="low"):
        super().__init__(model=model, verbosity=verbosity, reasoning=reasoning)
        self.prompts = prompts
        self.data = data
        self.metas = metas
        self.specialist_agent = specialist_agent

    def Orchestrate(
        self,
        type1,
        type2,
        specialistResponse,
        supervisorResponse,
        sp_vars,
        tries=0,
        printInfo=True
    ):

        # Limit retries
        if tries >= 3:
            return {
                "erro": "none",
                "proposta": "",
                "raciocinio": "LLM incapaz de trazer uma resposta válida"
            }


        # Validate specialist output
        if specialistResponse.get("erro") not in ["severo", "pequeno", "none"]:
            #print("Unknown Output From LLM:", specialistResponse)
            return {"error": "invalid specialist output"}

        # Run Self-Reflection
        selfReflectionResponse = self.specialist_agent.SelfReflection(
            source,
            machineTranslation,
            type1,
            type2,
            sp_vars,
            specialistResponse,
            printInfo
        )

        #if printInfo:
            #print("SelfReflection:", selfReflectionResponse)

        confianca = (
            selfReflectionResponse.get("confianca")
            or selfReflectionResponse.get("confiança")
        )

        # If confidence high → accept specialist answer
        if confianca == "alta":
            return specialistResponse

        # Supervisor agrees → accept specialist
        if (
            supervisorResponse.get("concordancia") == "concordo"
            and supervisorResponse.get("erro") == specialistResponse.get("erro")
        ):
            return specialistResponse

        # Supervisor disagrees → require debate
        sp_vars["severidade_su"] = supervisorResponse.get("erro")
        sp_vars["raciocinio_su"] = (
            supervisorResponse.get("raciocinio")
            or supervisorResponse.get("raciocínio")
        )
        sp_vars["proposta_su"] = supervisorResponse.get("proposta")

        # Build specialist reply for the debate
        replica = self.prompts.fill_template(
            self.data["evaluation"]["spreplica"],
            sp_vars
        )

        # DEBATE HISTORY
        specialistDebateHistory = []

        # system/developer rule
        specialistDebateHistory = self.appendHistory(
            specialistDebateHistory,
            "developer",
            self.data[type1][type2]
        )

        # original specialist answer
        specialistDebateHistory = self.appendHistory(
            specialistDebateHistory,
            "user",
            json.dumps(specialistResponse, ensure_ascii=False)
        )

        # supervisor's disagreement
        specialistDebateHistory = self.appendHistory(
            specialistDebateHistory,
            "assistant",
            json.dumps(supervisorResponse, ensure_ascii=False)
        )

        # final instruction
        final_prompt = (
            'Você concorda com a colocação de seu supervisor ou discorda? '
            'Se discordar, melhore seu raciocinio, proponha novamente um nivel de erro '
            'e proposta de tradução de en para pt-br em json com o formato: '
            '{"raciocinio":"seu raciocinio para a resposta","concordancia":"concordo|discordo", '
            '"erro":"severo|pequeno|none"}'
        )

        specialistDebateHistory = self.appendHistory(
            specialistDebateHistory,
            "user",
            final_prompt
        )

        #if printInfo:
            #print("FinalHistory:", specialistDebateHistory)

        # call final specialist answer
        specialistFinalResponse = self.callGPT(specialistDebateHistory)

        #if printInfo:
            #print("Final:", specialistFinalResponse)

        return specialistFinalResponse

# Calling All Agents

In [16]:
class CallAllAgents:
    def __init__(self, prompts_obj, data, metas, max_threads=4):
        self.prompts_obj = prompts_obj
        self.data = data
        self.metas = metas
        self.max_threads = max_threads
        self.semaphore = Semaphore(max_threads)
        self.results = {}
        self.lock = threading.Lock()

    def _process_agent(self, index, type1, type2, source, machineTranslation):
        """Processes one item of the list + one type1/type2 combination"""
        with self.semaphore:
            try:
                sp_vars = {
                    "pt_esp_error": self.metas[type1][type2],
                    "pt_sup_error": self.metas[type1]["pt_sup_error"],
                    "en_source": source,
                    "pt_machine_translation": machineTranslation,
                }

                # Specialist
                specialist_agent = Specialist(
                    prompts=self.prompts_obj,
                    data=self.data,
                    metas=self.metas,
                )

                specialistResponse = specialist_agent.SpecialistEvaluation(
                    source=source,
                    machineTranslation=machineTranslation,
                    type1=type1,
                    type2=type2,
                    printInfo=True,
                    sp_vars=sp_vars
                )

                self_reflection = specialist_agent.SelfReflection(
                    source=source,
                    machineTranslation=machineTranslation,
                    type1=type1,
                    type2=type2,
                    sp_vars=sp_vars,
                    specialistResponse=specialistResponse,
                    printInfo=True,
                )

                # Supervisor
                supervisor_agent = Supervisor(
                    prompts=self.prompts_obj,
                    data=self.data,
                )

                supervisorResponse = supervisor_agent.SupervisorEvaluation(
                    type1=type1,
                    sp_vars=sp_vars,
                    printInfo=True,
                )

                # Orchestrator
                orchestrator = Orchestrator(
                    prompts=self.prompts_obj,
                    data=self.data,
                    metas=self.metas,
                    specialist_agent=specialist_agent,
                )

                final_result = orchestrator.Orchestrate(
                    type1=type1,
                    type2=type2,
                    specialistResponse=specialistResponse,
                    supervisorResponse=supervisorResponse,
                    sp_vars=sp_vars,
                    printInfo=True,
                )

                # Store result indexed by item + type1 + type2
                with self.lock:
                    self.results[(index, type1, type2)] = final_result

            except Exception as e:
                print(f"Error at item {index} in {type1}/{type2}: {e}")
                with self.lock:
                    self.results[(index, type1, type2)] = f"Error: {e}"

    def call_all_agents(self, source, machineTranslation):
        """
        Receives LISTS of source + machineTranslation.
        type1/type2 combinations remain the same as your current logic.
        """
        threads = []

        for index in range(len(source)):
            src = source[index]
            mt = machineTranslation[index]

            for type1, subtypes in self.data.items():
                if type1 == "evaluation":
                    continue

                for type2, content in subtypes.items():
                    if type2 == "supervisor":
                        continue

                    thread = threading.Thread(
                        target=self._process_agent,
                        args=(index, type1, type2, src, mt),
                    )
                    threads.append(thread)
                    thread.start()

        # Wait for all
        for t in threads:
            t.join()

        return self.results

In [17]:
call = CallAllAgents(prompts_obj, data, metas)

results = call.call_all_agents(
    source=source,
    machineTranslation=machineTranslation
)

for key, value in results.items():
    print(key, value)

(0, 'linc', 'character') {'raciocinio': 'Não há sinais de corrupção de caracteres na comparação; o texto traduzido mantém acentuação e símbolos consistentes com a fonte em inglês, sem substituições de encoding aparentes. Portanto, erro = none.', 'erro': 'none', 'proposta': ''}
(0, 'term', 'inconsistent') {'raciocinio': "A inconsistência de terminologia ocorre quando termos equivalentes são usados de forma intercambiável dentro do mesmo texto. No entanto, neste trecho fornecido, há apenas uma correspondência consistente entre o termo em inglês e sua tradução: 'do that for free' → 'fazer isso de graça'. Não há variações de termos específicos substituídos por sinônimos diferentes no mesmo texto. Portanto, não há erro de inconsistência de terminologia.", 'erro': 'none', 'proposta': ''}
(0, 'linc', 'spelling') {'raciocinio': "A tradução parece correta ortograficamente em PT-BR. Não há erros de ortografia aparentes na frase traduzida: termos como 'Tom', 'ainda', 'está', 'disposto', 'a', 'faz

# Scores Our Metric

In [18]:
class ComputeScores:
    def __init__(self, results):
        self.results = results

    def _normalize_erro(self, erro):
        """
        Normalize values:
        - None -> "none"
        - "None", "NONE" -> "none"
        - everything becomes lowercase
        """
        if erro is None:
            return "none"

        if isinstance(erro, str):
            erro = erro.strip().lower()
            if erro == "":
                return "none"
            return erro

        return str(erro).strip().lower()

    def compute(self):
        """
        Count frequency of normalized erro values per index.
        Returns:
        {
            index: { erro_value: frequency }
        }
        """
        score_dict = {}

        for key, value in self.results.items():
            index = key[0]
            raw_erro = value.get("erro")
            normalized = self._normalize_erro(raw_erro)

            if index not in score_dict:
                score_dict[index] = {}

            score_dict[index][normalized] = score_dict[index].get(normalized, 0) + 1

        return score_dict

    def compute_final_scores(self, score_dict):
        """
        Compute final score starting from 100 and applying penalties:
        none     = -0 per item
        pequeno  = -2 per item
        severo   = -5 per item
        """
        penalty_map = {
            "none": 0,
            "pequeno": 2,
            "severo": 5
        }

        final_scores = {}

        for index, error_counts in score_dict.items():
            score = 100

            for erro_value, freq in error_counts.items():
                penalty = penalty_map.get(erro_value, 0)
                score -= penalty * freq

            final_scores[index] = max(score, 0)  # never negative

        return final_scores

    def save_final_scores_tsv(self, final_scores, filepath="final_scores.tsv"):
        """
        Saves final scores as a TSV file.
        Format:
        id    final_score
        """
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter="\t")
            writer.writerow(["id", "final_score"])

            for idx, score in final_scores.items():
                writer.writerow([idx, score])

        return filepath

    def print_scores(self, score_dict):
        for index, errors in score_dict.items():
            for erro_value, freq in errors.items():
                print(index, erro_value, freq)

In [19]:
scores = ComputeScores(results)

score_dict = scores.compute()
final_scores = scores.compute_final_scores(score_dict)

scores.save_final_scores_tsv(final_scores, "scores_hyp.tsv")

'scores_hyp.tsv'